# Election results

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

from datetime import datetime

import mapply
import pickle

In [ ]:
# Font
font_size = 16
fz = 1.5

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['CMU Serif Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = 18

mapply.init(
    n_workers=37,
    chunk_size=1,
    progressbar=True
)

## Load data

In [ ]:
import os
WORKING_DIR = os.environ.get("REPO_ROOT", os.path.abspath(".."))  # repo root (notebooks run from notebooks/)
DATA_DIR = f'{WORKING_DIR}/data'
IMG_DIR = f'{WORKING_DIR}/images'

### Load communes geometry and urbanization

In [ ]:
df_communes = pd.read_pickle(f'{DATA_DIR}/carto/communes.pkl')
df_communes = df_communes[['insee', 'urbanization_level']].copy()
df_communes.head(2)

### Parties details

In [ ]:
fd = open(f'{DATA_DIR}/parties_description/political_parties_description_europe_2019.json', 'r')
political_parties_description_europe_2019 = json.load(fd)
fd.close()

fd = open(f'{DATA_DIR}/parties_description/political_parties_description_europe_2024.json', 'r')
political_parties_description_europe_2024 = json.load(fd)
fd.close()

fd = open(f'{DATA_DIR}/parties_description/political_parties_description_legislative_2024.json', 'r')
political_parties_description_legislative_2024 = json.load(fd)
fd.close()

In [ ]:
for election in [political_parties_description_europe_2019,
                 political_parties_description_europe_2024,
                political_parties_description_legislative_2024]:
    print('Target Parties')
    for party in election:
        party_name = election[party]['name']
        print(f'{party_name} ({party})')
    print('')

### Election results

In [ ]:
def get_insee_code(departement, commune):
    return f'{str(departement).zfill(2)}{str(commune).zfill(3)}'

#### European Parlamentary Election - 2019

In [ ]:
df_european_parlamentary_election_2019 = pd.read_excel(f'{DATA_DIR}/elections/communes/2019/parlamentary_election/resultats-definitifs-par-commune.xls')
df_european_parlamentary_election_2019['insee'] = df_european_parlamentary_election_2019.apply(lambda x: get_insee_code(x['Code du département'], x['Code de la commune']), axis=1)

df_european_parlamentary_election_2019.head(2)

#### European Parlamentary Election - 2024

In [ ]:
dtype= {'Code commune': 'string'}

df_european_parlamentary_election_2024 = pd.read_csv(f'{DATA_DIR}/elections/communes/2024/parlamentary_election/resultats-definitifs-par-commune.csv', 
                               sep=';', dtype=dtype)
df_european_parlamentary_election_2024.rename(columns={'Code commune': 'insee'}, inplace=True)

df_european_parlamentary_election_2024.head(2)

In [ ]:
df_a = df_european_parlamentary_election_2024.merge(df_communes[['insee', 'urbanization_level']], on='insee', how='left')

total_inscrits = df_a['Inscrits'].sum()
total_voters = df_a['Votants'].sum()

df_election_eligible_voters_urban = df_a[df_a['urbanization_level'] != 'rural']
total_inscrits_urban = df_election_eligible_voters_urban['Inscrits'].sum()
total_voters_urban = df_election_eligible_voters_urban['Votants'].sum()

percentage_urban_inscrits = total_inscrits_urban / total_inscrits * 100
percentage_urban_voters = total_voters_urban / total_voters * 100
print(f'Percentage urban inscrits: {percentage_urban_inscrits:.4f} | Percentage urban voters: {percentage_urban_voters:.4f}')

## Get % of votes for target parties

In [ ]:
def get_european_parlamentry_parties_votes_columns(df_election, selected_parties, total_n_parties, offset, step, last_column):
    #### Separate target political parties from others parties
    columns = list(df_election.columns)

    all_votes_columns = [] # % Voix/Exp, % of votes/eligible voters
    for list_number in range(1, total_n_parties+1): 
        column_index = (list_number*step) + offset
        column_name = columns[column_index-last_column]
        all_votes_columns.append(column_name)

    target_votes_columns = []
    for party in selected_parties:
        if party == 'others':
            continue
        
        target_votes_column = selected_parties[party]['list_number']
        column_name = all_votes_columns[target_votes_column-1] # -1 because the list_number starts from 1
        target_votes_columns.append(column_name)

    other_parties_votes_columns = set(all_votes_columns) - set(target_votes_columns)
    other_parties_votes_columns = list(other_parties_votes_columns)

    return target_votes_columns, other_parties_votes_columns

In [ ]:
def other_parties_votes(row, other_parties_votes_columns):
    votes = 0
    for column in other_parties_votes_columns:
        votes += row[column]
    return votes

In [ ]:
def get_voters(row):
    inscrits = row['Inscrits']
    votants = row['Votants']
    return [inscrits, votants]

### European Parlamentary 2019

In [ ]:
target_votes_columns, other_parties_votes_columns = get_european_parlamentry_parties_votes_columns(df_european_parlamentary_election_2019, political_parties_description_europe_2019, 34, 17, 7, 0)
df_european_parlamentary_election_2019['others_votes'] = df_european_parlamentary_election_2019.apply(lambda x: other_parties_votes(x, other_parties_votes_columns), axis=1)
df_european_parlamentary_election_2019[['inscrits', 'voters']] = df_european_parlamentary_election_2019.apply(lambda x: get_voters(x), axis=1, result_type='expand')
df_european_parlamentary_election_2019 = df_european_parlamentary_election_2019[  ['insee'] + target_votes_columns + ['others_votes'] + ['inscrits', 'voters'] ]

renamed_columns = {}
for old_column, party in zip(target_votes_columns, political_parties_description_europe_2019):
    renamed_columns[old_column] = f'{party}_votes'

df_european_parlamentary_election_2019 = df_european_parlamentary_election_2019.rename(columns=renamed_columns)
european_parlamentary_votes_columns_2019 = list(df_european_parlamentary_election_2019.columns)[1:-2]
df_european_parlamentary_election_2019['check'] = df_european_parlamentary_election_2019.apply(lambda x: sum([x[column] for column in european_parlamentary_votes_columns_2019]), axis=1)
df_european_parlamentary_election_2019['check_without_other'] = df_european_parlamentary_election_2019.apply(lambda x: sum([x[column] for column in european_parlamentary_votes_columns_2019[:-1]]), axis=1)

df_european_parlamentary_election_2019.head(2)

### European Parlamentary 2024

In [ ]:
target_votes_columns, other_parties_votes_columns = get_european_parlamentry_parties_votes_columns(df_european_parlamentary_election_2024, political_parties_description_europe_2024, 38, 17, 8, 1)

df_european_parlamentary_election_2024[['inscrits', 'voters']] = df_european_parlamentary_election_2024.apply(lambda x: get_voters(x), axis=1, result_type='expand')

for column in target_votes_columns + other_parties_votes_columns:
    df_european_parlamentary_election_2024[column] = df_european_parlamentary_election_2024[column].str.replace('%', '').str.replace(',', '.').astype(float)

df_european_parlamentary_election_2024['others_votes'] = df_european_parlamentary_election_2024.apply(lambda x: other_parties_votes(x, other_parties_votes_columns), axis=1)
df_european_parlamentary_election_2024 = df_european_parlamentary_election_2024[  ['insee'] + target_votes_columns + ['others_votes'] + ['inscrits', 'voters'] ]

renamed_columns = {}
for old_column, party in zip(target_votes_columns, political_parties_description_europe_2024):
    renamed_columns[old_column] = f'{party}_votes'


df_european_parlamentary_election_2024 = df_european_parlamentary_election_2024.rename(columns=renamed_columns)
european_parlamentary_votes_columns_2024 = list(df_european_parlamentary_election_2024.columns)[1:-2]
df_european_parlamentary_election_2024['check'] = df_european_parlamentary_election_2024.apply(lambda x: sum([x[column] for column in european_parlamentary_votes_columns_2024]), axis=1)
df_european_parlamentary_election_2024['check_without_other'] = df_european_parlamentary_election_2024.apply(lambda x: sum([x[column] for column in european_parlamentary_votes_columns_2024[:-1]]), axis=1)

df_european_parlamentary_election_2024.head(2)

## Create elections object

In [ ]:
elections = {
    'europe_2019': {
        'name': 'European Parlamentary\nElection 2019',
        'color': 'tab:blue',
        'date': datetime(2019, 5, 26),
        'period_to_consider': (datetime(2019, 5, 19), datetime(2019, 5, 26)),
        # # May 19th, 2019 (included) to May 26th, 2019 (excluded)
        'parties_descriptions': political_parties_description_europe_2019,
        'results': df_european_parlamentary_election_2019
    },
    'europe_2024': {
        'name': 'European Parlamentary\nElection 2024',
        'color': 'tab:green',
        'date': datetime(2024, 6, 9),
        'period_to_consider': (datetime(2024, 6, 2), datetime(2024, 6, 9)),
        # # June 2nd, 2024 (included) to June 9th, 2024 the day of the election (excluded)
        'parties_descriptions': political_parties_description_europe_2024,
        'results': df_european_parlamentary_election_2024
    }
}

In [ ]:
fig, axs = plt.subplots(1, len(elections), figsize=(8*len(elections), 5))

bins = np.linspace(0, 100, 100)

for election_index, election in enumerate(elections):
    print(election)
    df_election_results = elections[election]['results']
    election_color = elections[election]['color']
    election_name = elections[election]['name']
    total_percentages = list(df_election_results['check'])

    ax = axs[election_index]
    ax.set_title(election_name)
    ax.hist(total_percentages, bins, alpha=0.95, color=election_color)
    ax.set_ylim(1, 1e5)
    ax.set_yscale('log')
    
plt.show()

### remove the communes with 0 participants

In [ ]:
for election_index, election in enumerate(elections):
    df_election_results = elections[election]['results'].copy()

    df_election_results = df_election_results[df_election_results['check'] > 90]
    df_election_results = df_election_results[df_election_results['check_without_other'] > 0]

    df_election_results.drop(columns=['check', 'check_without_other'], inplace=True)
    elections[election]['results'] = df_election_results.copy()

## Stats

### % of votes for major parties

In [ ]:
main_parties_columns

In [ ]:
bins = np.linspace(0, 100, 100)

fig, axs = plt.subplots(1, len(elections), figsize=(8*len(elections), 5))

for election_index, election in enumerate(elections):
    parties_details = elections[election]['parties_descriptions']
    df_election_results = elections[election]['results']
    election_color = elections[election]['color']
    election_name = elections[election]['name']
    main_parties_columns = list(df_election_results.columns)[1:-3]

    percentage_votes_main_parties = df_election_results[main_parties_columns].sum(axis=1).to_list()

    mean_percentage_votes_main_parties = np.mean(percentage_votes_main_parties)
    median_percentage_votes_main_parties = np.median(percentage_votes_main_parties)

    ax = axs[election_index]
    ax.set_title(election_name)
    
    ax.hist(percentage_votes_main_parties, bins, color=election_color, alpha=0.95)
    ax.axvline(median_percentage_votes_main_parties, color='g', linestyle='dashed', linewidth=1)
    ax.text(median_percentage_votes_main_parties-2, 1000, f'Median: {median_percentage_votes_main_parties:.2f}%', rotation=0, ha='right', va='bottom')
    ax.set_ylabel('Communes')
    ax.set_xlabel('Percentage of votes for main parties')
    ax.set_xlim(0, 100)
    ax.set_yscale('log')
    
plt.show()

### Ideology & Polarization

In [ ]:
for election_index, election in enumerate(elections):
    parties_details = elections[election]['parties_descriptions']
    main_parties_votes = [parties_details[party]['%votes'] for party in parties_details]
    main_parties_votes = np.array(main_parties_votes)

    main_parties_lr_score = [parties_details[party]['lr_score'] for party in parties_details]
    main_parties_lr_score = np.array(main_parties_lr_score)

    ideology = np.sum(main_parties_votes*main_parties_lr_score)/np.sum(main_parties_votes)
    polarization = np.sqrt(np.sum(main_parties_votes*((main_parties_lr_score-ideology)/5)**2))

    elections[election]['ideology'] = ideology
    elections[election]['polarization'] = polarization

In [ ]:
plt.figure(figsize=(8, 6))

for election_index, election in enumerate(elections):
    if election == 'legislative_2024':
        continue

    parties_details = elections[election]['parties_descriptions']
    df_election_results = elections[election]['results']
    color = elections[election]['color']

    ideology = elections[election]['ideology']
    polarization = elections[election]['polarization']
    
    plt.bar(0+(2*election_index-1)*.25, ideology, color=color, width=.45, alpha=0.75)
    plt.bar(2+(2*election_index-1)*.25, polarization, color=color, width=.45, alpha=0.75, hatch='//')

    plt.text(0+(2*election_index-1)*.25, ideology, f'{ideology:.2f}', rotation=0, ha='center', va='bottom')
    plt.text(2+(2*election_index-1)*.25, polarization, f'{polarization:.2f}', rotation=0, ha='center', va='bottom')


plt.grid(axis='y', linestyle='-', alpha=0.5)
plt.xticks([0, 2], ['Ideology', 'Polarization'])
plt.ylabel('Score')
plt.ylim(0, 10)

plt.savefig(f'{IMG_DIR}/ideology_polarization/ideology_polarization_france.pdf', bbox_inches='tight', dpi=200)
plt.show()

#### Polarization & Ideology per commune

In [ ]:
def compute_polarization_dalton(row, main_parties_columns, main_parties_lr_score, country_ideology):
    votes = row[main_parties_columns].to_numpy()
    polarization = np.sqrt(np.sum(votes*((main_parties_lr_score-country_ideology)/5)**2))
    return polarization

def compute_ideology(row, main_parties_columns, main_parties_lr_score):
    votes = row[main_parties_columns].to_numpy()
    ideology = np.sum(votes*main_parties_lr_score)/np.sum(votes)
    return ideology


polarization_funtions = {
    'polarization_dalton': compute_polarization_dalton,
}

In [ ]:
for election_index, election in enumerate(elections):

    parties_details = elections[election]['parties_descriptions']
    df_election_results = elections[election]['results'].copy()
    
    main_parties_lr_score = [parties_details[party]['lr_score'] for party in parties_details]
    main_parties_lr_score = np.array(main_parties_lr_score)

    main_parties_columns = list(df_election_results.columns)[1:-3]

    country_ideology = elections[election]['ideology']
    for polarization in polarization_funtions:
        polarization_function = polarization_funtions[polarization]
        df_election_results[polarization] = df_election_results.mapply(lambda x: polarization_function(x, main_parties_columns, main_parties_lr_score, country_ideology), axis=1)
    
    df_election_results['ideology'] = df_election_results.mapply(lambda x: compute_ideology(x, main_parties_columns, main_parties_lr_score), axis=1)
    
    elections[election]['results'] = df_election_results.copy()

#### Plot polarization and ideology distribution

In [ ]:
bins = np.linspace(0, 10, 150)

for polarization in polarization_funtions:

    plt.figure(figsize=(8, 6))
    for election_index, election in enumerate(elections):

        election_color = elections[election]['color']
        election_name = elections[election]['name']

        df_election_results = elections[election]['results'].copy()
        df_election_results_urbanization = df_election_results.merge(df_communes, on='insee', how='left')
        df_election_results_urbanization_urban = df_election_results_urbanization[df_election_results_urbanization['urbanization_level'] != 'rural']

        polarization_values = df_election_results_urbanization_urban[polarization]
        polarization_median = np.median(polarization_values)

        plt.hist(polarization_values, bins=bins, 
                                    color=election_color, 
                                    alpha=.85,
                                    label=f'{election_name}')
        
        median_color = 'tab:pink'
        if election == 'europe_2024':
            median_color = 'tab:purple'

plt.xlim(2, 8)

plt.ylabel('Communes')
plt.xlabel('Polarization')

plt.savefig(f'{IMG_DIR}/ideology_polarization/polarization_urban_comunes.pdf', bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
bins = np.linspace(0, 10, 150)

plt.figure(figsize=(8, 6))
for election_index, election in enumerate(elections):

    if election == 'legislative_2024':
        continue

    election_color = elections[election]['color']
    election_name = elections[election]['name']

    df_election_results = elections[election]['results'].copy()
    df_election_results_urbanization = df_election_results.merge(df_communes, on='insee', how='left')
    df_election_results_urbanization_urban = df_election_results_urbanization[df_election_results_urbanization['urbanization_level'] != 'rural']
    ideology_values = df_election_results_urbanization_urban['ideology']
    ideology_median = np.median(ideology_values)

    plt.hist(ideology_values, bins=bins, 
                                                color=election_color, 
                                                alpha=1-0.25*election_index,
                                                label=f'{election_name}')
    
    median_color = 'tab:pink'
    if election == 'europe_2024':
        median_color = 'tab:purple'
    

plt.xlim(3, 9)

plt.ylabel('Communes')
plt.xlabel('Ideology')
plt.savefig(f'{IMG_DIR}/ideology_polarization/ideology_urban_comunes.pdf', bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
plt.figure(figsize=(4, .5))
for election_index, election in enumerate(elections):

    if election == 'legislative_2024':
        continue
    
    election_color = elections[election]['color']
    election_name = elections[election]['name']
    election_year = elections[election]['date'].year

    plt.bar([0], [0], color=election_color, alpha=1-0.2*election_index,
                                            label=f'{election_year}')
    
plt.legend(loc='upper right', 
           ncol=2,
           fancybox=True,
           frameon=False,
           bbox_to_anchor=(1, 1.5))

plt.axis('off')
plt.savefig(f'{IMG_DIR}/ideology_polarization/legend.pdf', bbox_inches='tight', dpi=200)
plt.show()

## Save data

In [ ]:
pickle.dump(elections, open(f'{DATA_DIR}/elections/elections_communes.pkl', 'wb'))